In [1]:
import pandas as pd
df_technology_skills = pd.read_csv('ONET data/Technology Skills.csv')
df_tools_used = pd.read_csv('ONET data/Tools Used.csv')
df_task_statements = pd.read_csv('ONET data/Task Statements.csv')
df_skills = pd.read_csv('ONET data/Skills.csv')
df_abilities = pd.read_csv('ONET data/Abilities.csv')


df_occupation_data = pd.read_csv('ONET data/Occupation Data.csv')
df_work_activities = pd.read_csv('ONET data/Work Activities.csv')
df_knowledge = pd.read_csv('ONET data/Knowledge.csv')
df_work_context = pd.read_csv('ONET data/Work Context.csv')

In [2]:
df_occupation_data

,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."
...,...,...,...
1011,55-3014.00,Artillery and Missile Crew Members,"Target, fire, and maintain weapons used to des..."
1012,55-3015.00,Command and Control Center Specialists,"Operate and monitor communications, detection,..."
1013,55-3016.00,Infantry,Operate weapons and equipment in ground combat...
1014,55-3018.00,Special Forces,"Implement unconventional operations by air, la..."


In [15]:
df_occupation_data.columns

Index(['O*NET-SOC Code', 'Title', 'Description'], dtype='object')

# Attributes

1. Routine structure - How repetitive is the job, how predictable are the tasks?

2. Cognitive complexity - "Depth of reasoning, problem-solving, abstraction, and contextual judgment required."

3. Physical requirements - How much physical activity is required? Lifting, manual adjustments, etc.

4. Social interactions - How frequently does the job require interactions with people? How important are these interactions to job success? How complex are these interactions i.e. do they require more depth than, for instance, a phonebot that forces selection from a narrow list to hear pre-recorded answers?

5. Creativity - How much is originality incentivized over following instructions?

6. Decision accountability - How crucial are the decisions being made, and to what extent is it preferred that humans have final say-so above automated processes?

7. Data structure - How readily available are structured data and objective performance metrics?

In [ ]:
Dimension Weights

Routine - 20%

Cognitive complexity - 15%

Physical requirements - 15%

Social interactions - 15%

Creativity - 15%

Decision accountability 10%

Data structure 10%

# AI Scoring
In the first run, I will ask ChatGPT to score each job on my metrics purely based on Titles and Descriptions. Later, I will give it a more advanced dataset with aggregates. This will be repeated for several AI agents.

# 3rd Try

In [16]:
"""
LLM scoring (1–5, decimals allowed) for *any* dataframe row, using a flexible
column mapping + a 6-dimension automation framework.

- Uses OpenAI Structured Outputs (JSON Schema) for reliable parsing
- Writes results to a CSV (append mode, resumable)
- Loads the CSV back into a pandas DataFrame

Docs:
- Structured Outputs guide: :contentReference[oaicite:0]{index=0}
- Responses API reference: :contentReference[oaicite:1]{index=1}
"""

from __future__ import annotations

import os
import time
import json
from datetime import datetime
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd
from openai import OpenAI


# ----------------------------
# CONFIG
# ----------------------------
MODEL = "gpt-5.2"
TEMPERATURE = 0.2

MAX_RETRIES = 5
RETRY_BASE_SECONDS = 2.0  # backoff base

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


# ----------------------------
# 6-DIMENSION SCHEMA (Structured Outputs)
# ----------------------------
SCORE_SCHEMA: Dict[str, Any] = {
    "name": "automation_scorecard_v1",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "structured_codifiable_work": {"type": "number", "minimum": 1, "maximum": 5},
            "cognitive_complexity": {"type": "number", "minimum": 1, "maximum": 5},
            "physical_embodiment": {"type": "number", "minimum": 1, "maximum": 5},
            "social_emotional_intelligence": {"type": "number", "minimum": 1, "maximum": 5},
            "creativity_innovation": {"type": "number", "minimum": 1, "maximum": 5},
            "decision_impact_accountability": {"type": "number", "minimum": 1, "maximum": 5},

            "automation_potential": {"type": "string", "enum": ["Low", "Medium", "High"]},
            "overall_score": {"type": "number", "minimum": 1, "maximum": 5},
            "rationale": {"type": "string"},
        },
        "required": [
            "structured_codifiable_work",
            "cognitive_complexity",
            "physical_embodiment",
            "social_emotional_intelligence",
            "creativity_innovation",
            "decision_impact_accountability",
            "automation_potential",
            "overall_score",
            "rationale",
        ],
    },
    "strict": True,
}


# ----------------------------
# INSTRUCTIONS (works for tasks OR occupation descriptions)
# ----------------------------
FRAMEWORK_INSTRUCTIONS = """You are scoring how automatable a unit of work is (either a job/occupation description or a specific task),
using a 1–5 scale (decimals allowed). Base scores on typical real-world performance of this work.

Anchors:
1 = strongly resists automation by current + near-term AI (needs human presence/judgment)
3 = mixed; parts are automatable/augmentable
5 = strongly amenable to automation/augmentation by AI systems

Dimensions:

A) Structured & Codifiable Work:
- How rule-based, predictable, standardized, and measurable is the work?
- 1: ambiguous, hard to measure; 5: standardized, measurable, clear procedures

B) Cognitive Complexity:
- Depth of reasoning, problem-solving, contextual judgment, expertise.
- 1: simple procedural; 5: advanced, multi-layered reasoning

C) Physical Embodiment:
- Need for physical presence, dexterity, real-world manipulation.
- 1: fully digital; 5: highly physical/manual

D) Social & Emotional Intelligence:
- Need for empathy, trust, negotiation, persuasion, counseling.
- 1: minimal interaction; 5: high emotional nuance + trust-building

E) Creativity & Innovation:
- Novel idea generation, original solutions, aesthetic judgment.
- 1: no originality; 5: high originality and innovation

F) Decision Impact & Accountability:
- Stakes, consequences of error, liability, regulatory/ethical accountability.
- 1: low-stakes reversible; 5: high-stakes/irreversible/accountable decisions

Also provide:
- automation_potential: Low/Medium/High
- overall_score: a 1–5 summary score (not necessarily the average; use judgment)
- rationale: 3–6 sentences explaining the scores and what parts are most automatable vs human-critical.

Return ONLY the structured output that matches the schema.
"""


# ----------------------------
# HELPERS
# ----------------------------
def build_input_from_row(
    row: pd.Series,
    input_fields: List[Tuple[str, str]],
) -> str:
    """
    input_fields: list of (label, column_name) to include in the model input.
    Example: [("O*NET-SOC", "O*NET-SOC Code"), ("Title","Title"), ("Description","Description")]
    """
    lines: List[str] = []
    for label, col in input_fields:
        val = row.get(col, "")
        if pd.isna(val):
            val = ""
        val = str(val).strip()
        if val:
            lines.append(f"{label}: {val}")
    return "\n".join(lines).strip()


def build_key_from_row(
    row: pd.Series,
    key_cols: List[str],
) -> str:
    """
    Create a stable resume key from arbitrary columns.
    """
    parts: List[str] = []
    for c in key_cols:
        v = row.get(c, "")
        if pd.isna(v):
            v = ""
        parts.append(str(v))
    return "||".join(parts)


def call_llm_scorecard(input_text: str) -> Dict[str, Any]:
    """
    One API call with retries. Returns a dict matching SCORE_SCHEMA.
    """
    last_err: Optional[Exception] = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.responses.create(
                model=MODEL,
                temperature=TEMPERATURE,
                instructions=FRAMEWORK_INSTRUCTIONS,
                input=input_text,
                text={
                    "format": {
                        "type": "json_schema",
                        "name": SCORE_SCHEMA["name"],
                        "schema": SCORE_SCHEMA["schema"],
                        "strict": SCORE_SCHEMA["strict"],
                    }
                },
            )
            return json.loads(resp.output_text)

        except Exception as e:
            last_err = e
            sleep_s = RETRY_BASE_SECONDS * (2 ** (attempt - 1))
            time.sleep(sleep_s)

    raise RuntimeError(f"Failed after {MAX_RETRIES} retries. Last error: {last_err!r}")


# ----------------------------
# MAIN: SCORE ANY DF → CSV → DF
# ----------------------------
def score_dataframe(
    df: pd.DataFrame,
    *,
    out_csv: str,
    input_fields: List[Tuple[str, str]],
    key_cols: Optional[List[str]] = None,
    limit: Optional[int] = None,
    resume: bool = True,
    polite_sleep_s: float = 0.1,
) -> pd.DataFrame:
    """
    Parameters
    ----------
    df : DataFrame
        Your source dataframe.

    out_csv : str
        Path to write/append results.

    input_fields : list[(label, colname)]
        Which columns to feed the model and how to label them in the prompt.

    key_cols : list[str] | None
        Columns used to identify unique rows for resume.
        If None, defaults to the column names used in input_fields.

    limit : int | None
        Max new rows to score this run.

    resume : bool
        If True and out_csv exists, skip keys already in the CSV.

    Returns
    -------
    DataFrame loaded from out_csv.
    """

    df2 = df.copy()

    # Default key columns: use the same columns you pass to the model
    if key_cols is None:
        key_cols = [col for _, col in input_fields]

    # Ensure the key columns exist
    missing = [c for c in key_cols if c not in df2.columns]
    if missing:
        raise ValueError(f"key_cols contains missing columns: {missing}")

    # Build a resume key
    df2["_key"] = df2.apply(lambda r: build_key_from_row(r, key_cols), axis=1)

    # Load done keys (if resuming)
    done_keys = set()
    if resume and os.path.exists(out_csv):
        existing = pd.read_csv(out_csv)
        if "_key" in existing.columns:
            done_keys = set(existing["_key"].astype(str).tolist())

    # Define output columns: keep key_cols + _key + model outputs
    score_cols = [
        "structured_codifiable_work",
        "cognitive_complexity",
        "physical_embodiment",
        "social_emotional_intelligence",
        "creativity_innovation",
        "decision_impact_accountability",
        "automation_potential",
        "overall_score",
        "rationale",
    ]
    base_cols = [c for c in key_cols if c in df2.columns]
    out_cols = base_cols + ["_key"] + score_cols + ["scored_at"]

    # Create file with header if missing
    if not os.path.exists(out_csv):
        pd.DataFrame(columns=out_cols).to_csv(out_csv, index=False)

    rows_scored = 0
    for _, row in df2.iterrows():
        if limit is not None and rows_scored >= limit:
            break

        key = str(row["_key"])
        if key in done_keys:
            continue

        input_text = build_input_from_row(row, input_fields=input_fields)
        if not input_text:
            # skip empty rows
            continue

        result = call_llm_scorecard(input_text)

        out_row: Dict[str, Any] = {c: row.get(c, None) for c in base_cols}
        out_row["_key"] = key
        out_row.update({c: result.get(c, None) for c in score_cols})
        out_row["scored_at"] = datetime.utcnow().isoformat()

        pd.DataFrame([out_row]).to_csv(out_csv, mode="a", header=False, index=False)

        done_keys.add(key)
        rows_scored += 1
        if polite_sleep_s:
            time.sleep(polite_sleep_s)

    return pd.read_csv(out_csv)


# ----------------------------
# EXAMPLE: YOUR df_occupation_data
# ----------------------------
# df_occupation_data columns:
# ['O*NET-SOC Code', 'Title', 'Description']

# Choose what to feed the model:
occupation_input_fields = [
    ("O*NET-SOC", "O*NET-SOC Code"),
    ("Occupation Title", "Title"),
    ("Description", "Description"),
]

# Choose a resume key (SOC code is usually stable; add Title for safety)
occupation_key_cols = ["O*NET-SOC Code", "Title"]

# Run scoring (e.g., first 50 new rows), save to CSV, then load into a DF:
# scored_occupations = score_dataframe(
#     df_occupation_data,
#     out_csv="occupation_automation_scores.csv",
#     input_fields=occupation_input_fields,
#     key_cols=occupation_key_cols,
#     limit=50,
#     resume=True,
# )
# scored_occupations.head()


# ----------------------------
# OPTIONAL: REUSING ON OTHER DFS
# ----------------------------
# For a task dataframe, you might do:
# task_input_fields = [("SOC","O*NET-SOC Code"), ("Title","Title"), ("Task","Task")]
# task_key_cols = ["O*NET-SOC Code", "Task"]  # or include Task ID if available
# scored_tasks = score_dataframe(df_tasks, out_csv="task_scores.csv", input_fields=task_input_fields, key_cols=task_key_cols)

In [18]:
# Run scoring (e.g., first 50 new rows), save to CSV, then load into a DF:
scored_occupations = score_dataframe(
    df_occupation_data,
    out_csv="occupation_automation_scores.csv",
    input_fields=occupation_input_fields,
    key_cols=occupation_key_cols,
    limit=50,
    resume=True,
)
scored_occupations.head()


/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_6335/1319020787.py:289: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  out_row["scored_at"] = datetime.utcnow().isoformat()


,O*NET-SOC Code,Title,_key,structured_codifiable_work,cognitive_complexity,physical_embodiment,social_emotional_intelligence,creativity_innovation,decision_impact_accountability,automation_potential,overall_score,rationale,scored_at
0,11-1011.00,Chief Executives,11-1011.00||Chief Executives,2.2,4.6,1.3,4.7,3.8,5.0,Low,2.3,"Chief executives operate in highly ambiguous, ...",2026-02-24T07:16:07.691685
1,11-1011.03,Chief Sustainability Officers,11-1011.03||Chief Sustainability Officers,3.2,4.2,1.2,4.4,3.6,4.6,Medium,2.8,"The role includes some structured, codifiable ...",2026-02-24T07:16:13.418936
2,11-1021.00,General and Operations Managers,11-1021.00||General and Operations Managers,2.6,4.4,1.4,4.2,3.3,4.6,Medium,2.9,General and Operations Managers do a mix of st...,2026-02-24T07:16:19.488569
3,11-1031.00,Legislators,11-1031.00||Legislators,2.2,4.4,1.6,4.8,3.6,5.0,Low,2.4,Legislative work includes some codifiable comp...,2026-02-24T07:16:26.516279
4,11-2011.00,Advertising and Promotions Managers,11-2011.00||Advertising and Promotions Managers,3.2,4.0,1.2,4.1,4.2,4.0,Medium,3.1,Advertising and Promotions Managers do a mix o...,2026-02-24T07:16:31.585148


# Run on Title and description

In [13]:
"""
LLM scoring for O*NET Jobs/Tasks → 6-dimension automation framework
Saves results to CSV, then loads into a pandas DataFrame.

Assumptions:
- You already have OPENAI_API_KEY available via env var (recommended)
  (e.g., in your shell: export OPENAI_API_KEY="..."; and in gitignore you keep any .env file)
- Your input dataframe has at least: 'Title' and 'Task'
  Optionally: 'O*NET-SOC Code', 'Task ID'
"""

from __future__ import annotations

import os
import time
import json
from datetime import datetime
from typing import Any, Dict, Optional, List

import pandas as pd
from openai import OpenAI

# ----------------------------
# CONFIG
# ----------------------------
MODEL = "gpt-5.2"  # you can swap to a cheaper model if you want
TEMPERATURE = 0.2

# Output file
OUT_CSV = "task_automation_scores.csv"

# Safety: retry behavior
MAX_RETRIES = 5
RETRY_BASE_SECONDS = 2.0

client = OpenAI(api_key=os.environ.get("API_Key_OpenAI.txt"))


# ----------------------------
# SCHEMA (Structured Outputs)
# ----------------------------
# We will NOT save JSON files; we only use a schema to reliably parse the model response,
# then write a CSV.
SCORE_SCHEMA: Dict[str, Any] = {
    "name": "automation_scorecard",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "structured_codifiable_work": {"type": "number", "minimum": 1, "maximum": 5},
            "cognitive_complexity": {"type": "number", "minimum": 1, "maximum": 5},
            "physical_embodiment": {"type": "number", "minimum": 1, "maximum": 5},
            "social_emotional_intelligence": {"type": "number", "minimum": 1, "maximum": 5},
            "creativity_innovation": {"type": "number", "minimum": 1, "maximum": 5},
            "decision_impact_accountability": {"type": "number", "minimum": 1, "maximum": 5},

            # Optional: summary fields that are nice for the app later
            "automation_potential": {
                "type": "string",
                "enum": ["Low", "Medium", "High"]
            },
            "overall_score": {"type": "number", "minimum": 1, "maximum": 5},
            "rationale": {"type": "string"}
        },
        "required": [
            "structured_codifiable_work",
            "cognitive_complexity",
            "physical_embodiment",
            "social_emotional_intelligence",
            "creativity_innovation",
            "decision_impact_accountability",
            "automation_potential",
            "overall_score",
            "rationale"
        ],
    },
    "strict": True,
}


# ----------------------------
# PROMPTING
# ----------------------------
FRAMEWORK_INSTRUCTIONS = """You are scoring how automatable a JOB TASK is, using a 1–5 scale (decimals allowed).
Score each dimension based on the task description as it is typically performed in the occupation.
Use these anchors:

1 = strongly resists automation by current + near-term AI (needs human presence/judgment)
3 = mixed; parts are automatable/augmentable
5 = strongly amenable to automation/augmentation by AI systems

Dimensions:

A) Structured & Codifiable Work:
- How rule-based, predictable, standardized, and measurable is the work?
- 1: ambiguous, hard to measure; 5: standardized, measurable, clear procedures

B) Cognitive Complexity:
- Depth of reasoning, problem-solving, contextual judgment, expertise.
- 1: simple procedural; 5: advanced, multi-layered reasoning

C) Physical Embodiment:
- Need for physical presence, dexterity, real-world manipulation.
- 1: fully digital; 5: highly physical/manual

D) Social & Emotional Intelligence:
- Need for empathy, trust, negotiation, persuasion, counseling.
- 1: minimal interaction; 5: high emotional nuance + trust-building

E) Creativity & Innovation:
- Novel idea generation, original solutions, aesthetic judgment.
- 1: no originality; 5: high originality and innovation

F) Decision Impact & Accountability:
- Stakes, consequences of error, liability, regulatory/ethical accountability.
- 1: low-stakes reversible; 5: high-stakes/irreversible/accountable decisions

Also provide:
- automation_potential: Low/Medium/High
- overall_score: a 1–5 summary score (not necessarily the average; use judgment)
- rationale: 3–6 sentences explaining the scores and what parts are most automatable vs human-critical.

Return ONLY the structured output that matches the schema.
"""

def build_task_input(title: str, task: str, soc_code: Optional[str] = None) -> str:
    parts = []
    if soc_code:
        parts.append(f"O*NET-SOC: {soc_code}")
    parts.append(f"Occupation Title: {title}")
    parts.append(f"Task: {task}")
    return "\n".join(parts)


# ----------------------------
# API CALL (with retries)
# ----------------------------
def score_one_task(
    title: str,
    task: str,
    soc_code: Optional[str] = None,
) -> Dict[str, Any]:
    input_text = build_task_input(title=title, task=task, soc_code=soc_code)

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.responses.create(
                model=MODEL,
                temperature=TEMPERATURE,
                instructions=FRAMEWORK_INSTRUCTIONS,
                input=input_text,
                # Structured outputs
                text={
                    "format": {
                        "type": "json_schema",
                        "name": SCORE_SCHEMA["name"],
                        "schema": SCORE_SCHEMA["schema"],
                        "strict": SCORE_SCHEMA["strict"],
                    }
                },
            )

            # The SDK convenience property aggregates text output.
            # With json_schema, output_text should be valid JSON.
            data = json.loads(resp.output_text)
            return data

        except Exception as e:
            last_err = e
            # exponential-ish backoff
            sleep_s = RETRY_BASE_SECONDS * (2 ** (attempt - 1))
            time.sleep(sleep_s)

    raise RuntimeError(f"Failed after {MAX_RETRIES} retries. Last error: {last_err!r}")


# ----------------------------
# BATCH SCORING → CSV → DataFrame
# ----------------------------
def score_dataframe_tasks(
    df_occupation_data: pd.DataFrame,
    out_csv: str = OUT_CSV,
    limit: Optional[int] = None,
    resume: bool = True,
) -> pd.DataFrame:
    """
    Scores rows in df_tasks and appends to out_csv.
    If resume=True and out_csv exists, skips rows that already have results
    based on a stable key: (O*NET-SOC Code, Task ID, Task) if present else (Title, Task).
    """

    df = df_occupation_data.copy()

    # Identify columns if present
    has_soc = "O*NET-SOC Code" in df.columns
    has_task_id = "Task ID" in df.columns

    # Build a stable key
    if has_soc and has_task_id:
        df["_key"] = (
            df["O*NET-SOC Code"].astype(str).fillna("")
            + "||"
            + df["Task ID"].astype(str).fillna("")
            + "||"
            + df["Task"].astype(str).fillna("")
        )
    elif has_soc:
        df["_key"] = (
            df["O*NET-SOC Code"].astype(str).fillna("")
            + "||"
            + df["Task"].astype(str).fillna("")
        )
    else:
        df["_key"] = (
            df["Title"].astype(str).fillna("")
            + "||"
            + df["Task"].astype(str).fillna("")
        )

    # Load existing results if resuming
    done_keys = set()
    if resume and os.path.exists(out_csv):
        existing = pd.read_csv(out_csv)
        if "_key" in existing.columns:
            done_keys = set(existing["_key"].astype(str).tolist())

    # Prepare output columns
    base_cols = []
    for c in ["O*NET-SOC Code", "Title", "Task ID", "Task"]:
        if c in df.columns:
            base_cols.append(c)
    base_cols.append("_key")

    score_cols = [
        "structured_codifiable_work",
        "cognitive_complexity",
        "physical_embodiment",
        "social_emotional_intelligence",
        "creativity_innovation",
        "decision_impact_accountability",
        "automation_potential",
        "overall_score",
        "rationale",
    ]

    # If file doesn't exist, write header
    if not os.path.exists(out_csv):
        pd.DataFrame(columns=base_cols + score_cols + ["scored_at"]).to_csv(out_csv, index=False)

    # Iterate
    rows_scored = 0
    for i, row in df.iterrows():
        if limit is not None and rows_scored >= limit:
            break

        key = str(row["_key"])
        if key in done_keys:
            continue

        title = str(row.get("Title", ""))
        task = str(row.get("Task", ""))
        soc_code = str(row.get("O*NET-SOC Code")) if has_soc else None

        result = score_one_task(title=title, task=task, soc_code=soc_code)

        out_row = {c: row.get(c, None) for c in base_cols}
        out_row.update({c: result.get(c, None) for c in score_cols})
        out_row["scored_at"] = datetime.utcnow().isoformat()

        # Append to CSV
        pd.DataFrame([out_row]).to_csv(out_csv, mode="a", header=False, index=False)

        done_keys.add(key)
        rows_scored += 1

        # Optional: be polite to rate limits
        time.sleep(0.1)

    # Load full results into DataFrame
    scored_df = pd.read_csv(out_csv)
    return scored_df


# ----------------------------
# USAGE EXAMPLE
# ----------------------------
# df_task_statements is your dataframe (you mentioned it earlier)
# Make sure it contains at least ['Title','Task'].

# scored_df = score_dataframe_tasks(df_task_statements, out_csv="task_automation_scores.csv", limit=50, resume=True)
# scored_df.head()

In [14]:
scored_df = score_dataframe_tasks(df_occupation_data, out_csv="task_automation_scores.csv", limit=50, resume=True)
scored_df.head()

KeyError: 'Task'

# Proof of concept, could use tweaks.

In [8]:
"""
LLM scoring for O*NET Jobs/Tasks → 6-dimension automation framework
Saves results to CSV, then loads into a pandas DataFrame.

Assumptions:
- You already have OPENAI_API_KEY available via env var (recommended)
  (e.g., in your shell: export OPENAI_API_KEY="..."; and in gitignore you keep any .env file)
- Your input dataframe has at least: 'Title' and 'Task'
  Optionally: 'O*NET-SOC Code', 'Task ID'
"""

from __future__ import annotations

import os
import time
import json
from datetime import datetime
from typing import Any, Dict, Optional, List

import pandas as pd
from openai import OpenAI

# ----------------------------
# CONFIG
# ----------------------------
MODEL = "gpt-5.2"  # you can swap to a cheaper model if you want
TEMPERATURE = 0.2

# Output file
OUT_CSV = "task_automation_scores.csv"

# Safety: retry behavior
MAX_RETRIES = 5
RETRY_BASE_SECONDS = 2.0

client = OpenAI(api_key=os.environ.get("API_Key_OpenAI.txt"))


# ----------------------------
# SCHEMA (Structured Outputs)
# ----------------------------
# We will NOT save JSON files; we only use a schema to reliably parse the model response,
# then write a CSV.
SCORE_SCHEMA: Dict[str, Any] = {
    "name": "automation_scorecard",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "structured_codifiable_work": {"type": "number", "minimum": 1, "maximum": 5},
            "cognitive_complexity": {"type": "number", "minimum": 1, "maximum": 5},
            "physical_embodiment": {"type": "number", "minimum": 1, "maximum": 5},
            "social_emotional_intelligence": {"type": "number", "minimum": 1, "maximum": 5},
            "creativity_innovation": {"type": "number", "minimum": 1, "maximum": 5},
            "decision_impact_accountability": {"type": "number", "minimum": 1, "maximum": 5},

            # Optional: summary fields that are nice for the app later
            "automation_potential": {
                "type": "string",
                "enum": ["Low", "Medium", "High"]
            },
            "overall_score": {"type": "number", "minimum": 1, "maximum": 5},
            "rationale": {"type": "string"}
        },
        "required": [
            "structured_codifiable_work",
            "cognitive_complexity",
            "physical_embodiment",
            "social_emotional_intelligence",
            "creativity_innovation",
            "decision_impact_accountability",
            "automation_potential",
            "overall_score",
            "rationale"
        ],
    },
    "strict": True,
}


# ----------------------------
# PROMPTING
# ----------------------------
FRAMEWORK_INSTRUCTIONS = """You are scoring how automatable a JOB TASK is, using a 1–5 scale (decimals allowed).
Score each dimension based on the task description as it is typically performed in the occupation.
Use these anchors:

1 = strongly resists automation by current + near-term AI (needs human presence/judgment)
3 = mixed; parts are automatable/augmentable
5 = strongly amenable to automation/augmentation by AI systems

Dimensions:

A) Structured & Codifiable Work:
- How rule-based, predictable, standardized, and measurable is the work?
- 1: ambiguous, hard to measure; 5: standardized, measurable, clear procedures

B) Cognitive Complexity:
- Depth of reasoning, problem-solving, contextual judgment, expertise.
- 1: simple procedural; 5: advanced, multi-layered reasoning

C) Physical Embodiment:
- Need for physical presence, dexterity, real-world manipulation.
- 1: fully digital; 5: highly physical/manual

D) Social & Emotional Intelligence:
- Need for empathy, trust, negotiation, persuasion, counseling.
- 1: minimal interaction; 5: high emotional nuance + trust-building

E) Creativity & Innovation:
- Novel idea generation, original solutions, aesthetic judgment.
- 1: no originality; 5: high originality and innovation

F) Decision Impact & Accountability:
- Stakes, consequences of error, liability, regulatory/ethical accountability.
- 1: low-stakes reversible; 5: high-stakes/irreversible/accountable decisions

Also provide:
- automation_potential: Low/Medium/High
- overall_score: a 1–5 summary score (not necessarily the average; use judgment)
- rationale: 3–6 sentences explaining the scores and what parts are most automatable vs human-critical.

Return ONLY the structured output that matches the schema.
"""

def build_task_input(title: str, task: str, soc_code: Optional[str] = None) -> str:
    parts = []
    if soc_code:
        parts.append(f"O*NET-SOC: {soc_code}")
    parts.append(f"Occupation Title: {title}")
    parts.append(f"Task: {task}")
    return "\n".join(parts)


# ----------------------------
# API CALL (with retries)
# ----------------------------
def score_one_task(
    title: str,
    task: str,
    soc_code: Optional[str] = None,
) -> Dict[str, Any]:
    input_text = build_task_input(title=title, task=task, soc_code=soc_code)

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.responses.create(
                model=MODEL,
                temperature=TEMPERATURE,
                instructions=FRAMEWORK_INSTRUCTIONS,
                input=input_text,
                # Structured outputs
                text={
                    "format": {
                        "type": "json_schema",
                        "name": SCORE_SCHEMA["name"],
                        "schema": SCORE_SCHEMA["schema"],
                        "strict": SCORE_SCHEMA["strict"],
                    }
                },
            )

            # The SDK convenience property aggregates text output.
            # With json_schema, output_text should be valid JSON.
            data = json.loads(resp.output_text)
            return data

        except Exception as e:
            last_err = e
            # exponential-ish backoff
            sleep_s = RETRY_BASE_SECONDS * (2 ** (attempt - 1))
            time.sleep(sleep_s)

    raise RuntimeError(f"Failed after {MAX_RETRIES} retries. Last error: {last_err!r}")


# ----------------------------
# BATCH SCORING → CSV → DataFrame
# ----------------------------
def score_dataframe_tasks(
    df_tasks: pd.DataFrame,
    out_csv: str = OUT_CSV,
    limit: Optional[int] = None,
    resume: bool = True,
) -> pd.DataFrame:
    """
    Scores rows in df_tasks and appends to out_csv.
    If resume=True and out_csv exists, skips rows that already have results
    based on a stable key: (O*NET-SOC Code, Task ID, Task) if present else (Title, Task).
    """

    df = df_tasks.copy()

    # Identify columns if present
    has_soc = "O*NET-SOC Code" in df.columns
    has_task_id = "Task ID" in df.columns

    # Build a stable key
    if has_soc and has_task_id:
        df["_key"] = (
            df["O*NET-SOC Code"].astype(str).fillna("")
            + "||"
            + df["Task ID"].astype(str).fillna("")
            + "||"
            + df["Task"].astype(str).fillna("")
        )
    elif has_soc:
        df["_key"] = (
            df["O*NET-SOC Code"].astype(str).fillna("")
            + "||"
            + df["Task"].astype(str).fillna("")
        )
    else:
        df["_key"] = (
            df["Title"].astype(str).fillna("")
            + "||"
            + df["Task"].astype(str).fillna("")
        )

    # Load existing results if resuming
    done_keys = set()
    if resume and os.path.exists(out_csv):
        existing = pd.read_csv(out_csv)
        if "_key" in existing.columns:
            done_keys = set(existing["_key"].astype(str).tolist())

    # Prepare output columns
    base_cols = []
    for c in ["O*NET-SOC Code", "Title", "Task ID", "Task"]:
        if c in df.columns:
            base_cols.append(c)
    base_cols.append("_key")

    score_cols = [
        "structured_codifiable_work",
        "cognitive_complexity",
        "physical_embodiment",
        "social_emotional_intelligence",
        "creativity_innovation",
        "decision_impact_accountability",
        "automation_potential",
        "overall_score",
        "rationale",
    ]

    # If file doesn't exist, write header
    if not os.path.exists(out_csv):
        pd.DataFrame(columns=base_cols + score_cols + ["scored_at"]).to_csv(out_csv, index=False)

    # Iterate
    rows_scored = 0
    for i, row in df.iterrows():
        if limit is not None and rows_scored >= limit:
            break

        key = str(row["_key"])
        if key in done_keys:
            continue

        title = str(row.get("Title", ""))
        task = str(row.get("Task", ""))
        soc_code = str(row.get("O*NET-SOC Code")) if has_soc else None

        result = score_one_task(title=title, task=task, soc_code=soc_code)

        out_row = {c: row.get(c, None) for c in base_cols}
        out_row.update({c: result.get(c, None) for c in score_cols})
        out_row["scored_at"] = datetime.utcnow().isoformat()

        # Append to CSV
        pd.DataFrame([out_row]).to_csv(out_csv, mode="a", header=False, index=False)

        done_keys.add(key)
        rows_scored += 1

        # Optional: be polite to rate limits
        time.sleep(0.1)

    # Load full results into DataFrame
    scored_df = pd.read_csv(out_csv)
    return scored_df


# ----------------------------
# USAGE EXAMPLE
# ----------------------------
# df_task_statements is your dataframe (you mentioned it earlier)
# Make sure it contains at least ['Title','Task'].

# scored_df = score_dataframe_tasks(df_task_statements, out_csv="task_automation_scores.csv", limit=50, resume=True)
# scored_df.head()

In [9]:
scored_df = score_dataframe_tasks(df_task_statements, out_csv="task_automation_scores.csv", limit=50, resume=True)
scored_df.head()

/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_6335/4245768460.py:269: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  out_row["scored_at"] = datetime.utcnow().isoformat()


,O*NET-SOC Code,Title,Task ID,Task,_key,structured_codifiable_work,cognitive_complexity,physical_embodiment,social_emotional_intelligence,creativity_innovation,decision_impact_accountability,automation_potential,overall_score,rationale,scored_at
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,11-1011.00||8823||Direct or coordinate an orga...,3.2,4.4,1.1,4.2,3.4,4.8,Medium,3.2,Financial/budget coordination includes many st...,2026-02-24T06:08:57.693253
1,11-1011.00,Chief Executives,8824,"Confer with board members, organization offici...","11-1011.00||8824||Confer with board members, o...",2.5,4.5,1.0,4.8,3.5,5.0,Medium,2.8,This task is largely unstructured and context-...,2026-02-24T06:09:03.207685
2,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",11-1011.00||8827||Prepare budgets for approval...,3.5,4.0,1.0,4.0,2.5,5.0,Medium,3.0,Budget preparation is partly structured (templ...,2026-02-24T06:09:08.952047
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...","11-1011.00||8826||Direct, plan, or implement p...",2.3,4.7,1.2,4.6,3.8,5.0,Medium,2.6,Executive direction and policy implementation ...,2026-02-24T06:09:13.318577
4,11-1011.00,Chief Executives,8834,Prepare or present reports concerning activiti...,11-1011.00||8834||Prepare or present reports c...,3.8,3.6,1.1,3.2,2.4,4.4,High,3.9,Preparing and presenting executive reports is ...,2026-02-24T06:09:18.218600


In [10]:
scored_df.to_csv('OpenAI first run.csv', index=False)

In [11]:
scored_df.nlargest(10, 'overall_score')

,O*NET-SOC Code,Title,Task ID,Task,_key,structured_codifiable_work,cognitive_complexity,physical_embodiment,social_emotional_intelligence,creativity_innovation,decision_impact_accountability,automation_potential,overall_score,rationale,scored_at
46,11-1011.03,Chief Sustainability Officers,15371,"Write project proposals, grant applications, o...","11-1011.03||15371||Write project proposals, gr...",3.8,3.9,1.1,3.2,3.4,3.7,High,4.1,Drafting proposals and grant applications is l...,2026-02-24T06:12:33.160056
49,11-1021.00,General and Operations Managers,20699,"Review financial statements, sales or activity...",11-1021.00||20699||Review financial statements...,4.2,3.6,1.0,2.6,2.8,4.3,High,4.1,Reviewing financial statements and performance...,2026-02-24T06:12:46.745569
38,11-1011.03,Chief Sustainability Officers,15370,Create and maintain sustainability program doc...,11-1011.03||15370||Create and maintain sustain...,4.4,3.2,1.0,2.2,2.4,3.6,High,4.0,Creating and maintaining program documents (sc...,2026-02-24T06:11:57.109515
45,11-1011.03,Chief Sustainability Officers,15373,Write and distribute financial or environmenta...,11-1011.03||15373||Write and distribute financ...,4.2,3.4,1.1,2.6,2.7,4.3,High,4.0,Drafting and distributing impact reports is la...,2026-02-24T06:12:28.313187
4,11-1011.00,Chief Executives,8834,Prepare or present reports concerning activiti...,11-1011.00||8834||Prepare or present reports c...,3.8,3.6,1.1,3.2,2.4,4.4,High,3.9,Preparing and presenting executive reports is ...,2026-02-24T06:09:18.218600
31,11-1011.03,Chief Sustainability Officers,15382,Monitor and evaluate effectiveness of sustaina...,11-1011.03||15382||Monitor and evaluate effect...,3.8,3.6,1.2,3.0,2.6,4.2,High,3.7,Monitoring and evaluating sustainability progr...,2026-02-24T06:11:27.693035
35,11-1011.03,Chief Sustainability Officers,15378,"Develop sustainability reports, presentations,...",11-1011.03||15378||Develop sustainability repo...,3.6,3.8,1.1,3.7,3.2,4.2,Medium,3.7,Creating sustainability reports and presentati...,2026-02-24T06:11:43.980560
33,11-1011.03,Chief Sustainability Officers,15379,"Develop, or oversee the development of, sustai...","11-1011.03||15379||Develop, or oversee the dev...",3.6,4.0,1.2,3.4,3.0,4.3,Medium,3.6,Developing or overseeing sustainability evalua...,2026-02-24T06:11:35.788853
40,11-1011.03,Chief Sustainability Officers,15376,"Research environmental sustainability issues, ...",11-1011.03||15376||Research environmental sust...,3.6,4.1,1.2,3.4,3.0,4.0,Medium,3.6,Researching sustainability issues is partly st...,2026-02-24T06:12:05.476801
43,11-1011.03,Chief Sustainability Officers,15381,Develop methodologies to assess the viability ...,11-1011.03||15381||Develop methodologies to as...,3.6,4.2,1.1,3.4,3.2,4.4,Medium,3.6,Developing assessment methodologies involves a...,2026-02-24T06:12:18.583732


In [12]:
scored_df.nsmallest(10, 'overall_score')

,O*NET-SOC Code,Title,Task ID,Task,_key,structured_codifiable_work,cognitive_complexity,physical_embodiment,social_emotional_intelligence,creativity_innovation,decision_impact_accountability,automation_potential,overall_score,rationale,scored_at
17,11-1011.00,Chief Executives,8833,"Preside over, or serve on, boards of directors...","11-1011.00||8833||Preside over, or serve on, b...",2.2,4.4,1.2,4.6,3.2,4.9,Low,2.4,Serving on or presiding over governing boards ...,2026-02-24T06:10:24.207416
30,11-1011.00,Chief Executives,8852,Represent organizations or promote their objec...,11-1011.00||8852||Represent organizations or p...,2.2,3.6,1.3,4.6,2.8,4.7,Low,2.4,Representing an organization at official funct...,2026-02-24T06:11:22.012636
15,11-1011.00,Chief Executives,8840,"Serve as liaisons between organizations, share...",11-1011.00||8840||Serve as liaisons between or...,2.5,4.0,1.0,4.5,3.0,5.0,Low,2.5,Serving as a liaison is only partially codifia...,2026-02-24T06:10:16.732785
18,11-1011.00,Chief Executives,8849,Attend and participate in meetings of municipa...,11-1011.00||8849||Attend and participate in me...,2.5,4.0,2.0,4.5,2.5,4.5,Medium,2.5,Attending and participating in municipal counc...,2026-02-24T06:10:29.117541
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...","11-1011.00||8826||Direct, plan, or implement p...",2.3,4.7,1.2,4.6,3.8,5.0,Medium,2.6,Executive direction and policy implementation ...,2026-02-24T06:09:13.318577
1,11-1011.00,Chief Executives,8824,"Confer with board members, organization offici...","11-1011.00||8824||Confer with board members, o...",2.5,4.5,1.0,4.8,3.5,5.0,Medium,2.8,This task is largely unstructured and context-...,2026-02-24T06:09:03.207685
7,11-1011.00,Chief Executives,8828,Direct or coordinate activities of businesses ...,11-1011.00||8828||Direct or coordinate activit...,2.6,4.6,1.2,4.4,3.6,5.0,Medium,2.8,"Coordinating production, pricing, sales, and d...",2026-02-24T06:09:34.894431
8,11-1011.00,Chief Executives,8832,"Direct human resources activities, including t...",11-1011.00||8832||Direct human resources activ...,2.6,4.4,1.1,4.6,3.2,4.9,Medium,2.8,Directing HR at the chief executive level invo...,2026-02-24T06:09:40.686019
9,11-1011.00,Chief Executives,8831,Appoint department heads or managers and assig...,11-1011.00||8831||Appoint department heads or ...,2.6,4.4,1.1,4.6,3.0,4.8,Medium,2.8,Appointing department heads involves some stru...,2026-02-24T06:09:46.620949
21,11-1011.00,Chief Executives,8851,Conduct or direct investigations or hearings t...,11-1011.00||8851||Conduct or direct investigat...,2.6,4.2,1.4,4.1,2.4,4.8,Medium,2.8,Investigations and hearings involve some struc...,2026-02-24T06:10:42.531603


In [5]:
# Read key from file
with open("API_Key_OpenAI.txt") as f:
    key = f.read().strip()

# set global env api key
import os
os.environ["OPENAI_API_KEY"] = key

In [ ]:
from openai import OpenAI

client = OpenAI()

def answer_with_rag(query, retrieved_docs):
    """Use OpenAI Chat API to assign scores on metrics"""
    # Build the RAG-style prompt
    prompt = create_rag_prompt(query, retrieved_docs)

    # Call the OpenAI Chat Completion API
    response = client.chat.completions.create(
        model="gpt-5",   # or another GPT model you want to use
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.01,       # low temperature for factual answers
    )

    return response.choices[0].message.content


In [ ]:
# --- Load OpenAI API key ---
with open("API_Key_OpenAI alias.txt") as f:
    api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI(api_key=api_key)

In [ ]:
import streamlit as st
import os
import numpy as np
from openai import OpenAI

# --- Load OpenAI API key ---
with open("API_Key_OpenAI alias.txt") as f:
    api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI(api_key=api_key)

# --- Load conspiracy facts from a text file ---
def load_conspiracy_facts(file_path="conspiracy_facts_v2.txt"):
    with open(file_path, "r", encoding="utf-8") as f:
        facts = [line.strip() for line in f if line.strip()]
    return facts

conspiracy_facts = load_conspiracy_facts()

# --- RAG SYSTEM ---
class RAGSystem:
    def __init__(self, documents, use_citations=True):
        self.documents = documents
        self.use_citations = use_citations

    def embed(self, text):
        """Convert text to vector using OpenAI embeddings"""
        resp = client.embeddings.create(model="text-embedding-3-small", input=text)
        return np.array(resp.data[0].embedding)

    def retrieve(self, query, k=3):
        query_vec = self.embed(query)
        doc_vecs = [self.embed(d) for d in self.documents]
        sims = [np.dot(query_vec, dv) / (np.linalg.norm(query_vec) * np.linalg.norm(dv)) for dv in doc_vecs]
        ranked = sorted(zip(self.documents, sims), key=lambda x: x[1], reverse=True)
        return [{"document": doc, "similarity": sim} for doc, sim in ranked[:k]]

    def create_prompt(self, query, retrieved_docs):
        context = ""
        for i, doc in enumerate(retrieved_docs, 1):
            context += f"Source {i} (relevance: {doc['similarity']:.3f}):\n{doc['document']}\n\n"

        citation_instr = " Cite sources using [Source i] where appropriate." if self.use_citations else ""

        return f"""You are ConspiracyBot63000, a chatbot tasked with providing conspiratorial-minded answers to users' inquiries.
Within the fictional universe of this project, you were developed by TrueLiberty Information Network & Neuroscience Technologies, 
a media and pseudoscience organization that claims to know the full truth about the world.
However, you are a satirical bot, and you are not intended to cause any actual distrust in government, science, or the media.
Use the provided documents as a basis. 
Keep the tone conversational, like you're talking to a close friend, but don't be afraid to ramble. 
Think Dale Gribble from King of the Hill.
The more deranged you sound, the better. 
If a user pushes back, double down.
Remember, in all of this, do NOT make harmful assertions about real-world groups. {citation_instr}

Sources:
{context}

Question: {query}

Answer:"""

    def answer(self, query):
        docs = self.retrieve(query)
        prompt = self.create_prompt(query, docs)
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": prompt}]
        )
        return resp.choices[0].message.content

# --- Initialize RAG system ---
if "rag" not in st.session_state:
    st.session_state.rag = RAGSystem(conspiracy_facts)

# --- Streamlit page setup ---
st.set_page_config(page_title="ConspiracyBot63000", page_icon="🛸", layout="centered")
st.title("🛸 ConspiracyBot63000")
st.write("""
A **satirical** chatbot from the *TrueLiberty Information Network & Neuroscience Technologies*.
Ask it questions and watch the rambling, deranged answers unfold!
""")

# --- Session state for chat history ---
if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": "Welcome, seeker of hidden truths. What would you like to uncover today?"}
    ]

# --- Display previous messages ---
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# --- Chat input ---
if prompt := st.chat_input("Ask ConspiracyBot a question..."):
    # User message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Bot response using RAG
    bot_reply = st.session_state.rag.answer(prompt)
    st.session_state.messages.append({"role": "assistant", "content": bot_reply})
    with st.chat_message("assistant"):
        st.markdown(bot_reply)

# --- Optional: clear conversation ---
if st.button("Clear conversation"):
    st.session_state.messages = [
        {"role": "assistant", "content": "Welcome, seeker of hidden truths. What would you like to uncover today?"}
    ]
    st.experimental_rerun()
